# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/Prop_aneuploid/output"

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/Figures/Prop_aneuploid/output/EXP71_72_72_perAneuploid.csv")

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.176

settheme <- theme_minimal() +
  theme(
    text = element_text(family = "sans", size = FONT.SIZE),
    panel.background = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(colour = "black"),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 0),
    legend.position = "right",
    title = element_text(colour = "black", size = FONT.SIZE),
    plot.title = element_text(size = FONT.SIZE, face = "plain")
  )

In [ ]:
color_condition = c("Control" = "#7BBA56", 
               "Reversine" = "#87549B", 
               "Mosaic" = "#E8973E")

col_Annexin= "#4db2cb"

col_ZVAD = c("ZVAD+" = "#6a6969", 
               "ZVAD-" = "#bbbbbb")

col_GFP = "#7BBA56"
col_RFP = "#cb377c"

col_condition2 = c("control" = "#7BBA56", 
               "mosaic_fixed" = "#d45902", 
               "mosaic" = "#E8973E")

col_structure= c("developed" = "#157AFF",  
        "small.cavity" ="#F09938" , 
        "failed" = "#fe941b")

## 1. Extract summary files

In [ ]:
merged_df <- read_csv(analysis_summary_files)


In [ ]:
head(merged_df)

In [ ]:
colnames(merged_df)

In [ ]:
unique(merged_df$per_reversine)

In [ ]:
unique(merged_df$EXP)

In [ ]:
df_sample = merged_df

# Plot 

### A) Proportion of structures 

In [ ]:

order_sample <- c(
'0%','25%','50%','75%','100%')

df_sample <- df_sample %>%
  mutate(per_reversine = factor(per_reversine, levels = order_sample))

In [ ]:
title = "proportion developed"
w <- 2
h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = per_reversine , y = proportion_developed)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.8, width = 0.6, fill = "#157AFF") +   # error bars
    geom_jitter(
      aes(fill = per_reversine),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.9, 
      color = "#505150"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "#505150")+
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))
      #facet_wrap( ~ condition_2) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "proportion developed"
w <- 1.5
h <- 1.5
options(repr.plot.width = w, repr.plot.height = h)
  
p = ggplot(df_sample, aes(x = per_reversine, y = proportion_developed, group = 1)) +
  
  # raw points
  geom_jitter(
    color = "#505150",
    position = position_jitter(width = 0.05, height = 0),
    size = 0.5,
    alpha = 0.9
  ) +
  
  # mean line
  stat_summary(
    fun = mean,
    geom = "line",
    linewidth = LINE.W,
    color = "#157AFF"
  ) +
  
  # mean points
  stat_summary(
    fun = mean,
    geom = "point",
    size = 0.5,
    color = "#157AFF"
  ) +
  
  # mean ± SE
  stat_summary(
    fun.data = mean_se,
    geom = "errorbar",
    width = 0.1,
    linewidth = LINE.W,
    color = "#505150"
  ) +
  
  labs(
    title = title,
    y = "proportion developed",
    x = "% reversine-treated cells"
  ) +
  settheme +
  theme(
    #axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none",
    legend.key.size = unit(0.3, "cm")
  ) +
  scale_y_continuous(
    limits = c(0, 1.1),
    expand = c(0, 0)
  )

ggsave(
  file.path(out_dir, sprintf("A_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:
df_sample <- df_sample %>%
  mutate(
    per_reversine_num = parse_number(as.character(per_reversine)),
    proportion_developed = as.numeric(proportion_developed)
  )

cor.test(
  df_sample$per_reversine_num,
  df_sample$proportion_developed,
  method = "spearman",
  exact = FALSE
)

In [ ]:
df_exp <- df_sample %>%
  group_by(EXP, per_reversine) %>%
  summarise(
    developed = sum(developed),
    failed = sum(failed),
    proportion_developed = developed/(developed+failed),
    .groups = "drop"
  )

In [ ]:
df_sample <- df_sample %>%
  mutate(
    per_reversine_num = parse_number(as.character(per_reversine)),
    developed = as.integer(developed),
    failed = as.integer(failed)
  )

m <- glm(cbind(developed, failed) ~ per_reversine_num,
         family = quasibinomial, data = df_sample)
summary(m)

In [ ]:
df_exp

In [ ]:
title = "proportion developed"
w <- 1.5
h <- 1.5
options(repr.plot.width = w, repr.plot.height = h)
  
p = ggplot(df_exp, aes(x = per_reversine, y = proportion_developed, group = 1)) +
  
  # raw points
  geom_jitter(
    color = "#505150",
    position = position_jitter(width = 0.05, height = 0),
    size = 0.5,
    alpha = 0.9
  ) +
  
  # mean line
  stat_summary(
    fun = mean,
    geom = "line",
    linewidth = LINE.W,
    color = "#157AFF"
  ) +
  
  # mean points
  stat_summary(
    fun = mean,
    geom = "point",
    size = 0.5,
    color = "#157AFF"
  ) +
  
  # mean ± SE
  stat_summary(
    fun.data = mean_se,
    geom = "errorbar",
    width = 0.1,
    linewidth = LINE.W,
    color = "#505150"
  ) +
  
  labs(
    title = title,
    y = "proportion developed",
    x = "% reversine-treated cells"
  ) +
  settheme +
  theme(
    #axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none",
    legend.key.size = unit(0.3, "cm")
  ) +
  scale_y_continuous(
    limits = c(0, 1.1),
    expand = c(0, 0)
  )

ggsave(
  file.path(out_dir, sprintf("A_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:
library(dplyr)
library(readr)

df_exp <- df_exp %>%
  mutate(
    EXP = factor(EXP),
    per_reversine_num = parse_number(as.character(per_reversine)),
    per_reversine_step = per_reversine_num / 25
  )

model <- glm(
  cbind(developed, failed) ~ per_reversine_step + EXP,
  family = binomial(link = "logit"),
  data = df_exp
)

summary(model)

# Overall test of the reversine trend
drop1(model, test = "F")

In [ ]:
df_exp <- df_sample %>%
  group_by(per_reversine) %>%
  summarise(
    developed = sum(developed),
    failed = sum(failed),
    proportion_developed = developed/(developed+failed),
    .groups = "drop"
  )

In [ ]:
df_exp 

In [ ]:
7